In [5]:
import torch
import torch.nn as nn
from torch.nn import functional as F



batch_size = 64 
block_size = 256 
max_iters = 3000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
# ------------


torch.manual_seed(1337)

#读取数据
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()


#提取文本中出现的字符并排序，获得字符种类数量
chars = sorted(list(set(text)))
vocab_size = len(chars)

#建立双向词表，实现字符与数值之间的转换
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
#编码，st->i
encode = lambda s: [stoi[c] for c in s]
#解码，i->st
decode = lambda l: ''.join([itos[i] for i in l])

#获取数据，切分为数据集与验证集
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) 
train_data = data[:n]
val_data = data[n:]

#数据切分出x,y；做好block/batch切分，方便训练
def get_batch(split):
    #选定使用的数据集类型
    data = train_data if split == 'train' else val_data
    #在data序列中选出batch_size个位置，往后抽出block_size个block
    #每个block有block_size个数值，len(data) - block_size,防止序列溢出
    ix = torch.randint(len(data) - block_size, (batch_size,))
    #使用stack,凭空创造Batch维度，把每个一维向量当成矩阵的一行叠起来
    #切出block的对应输入数值(Batch_size, block_size)
    x = torch.stack([data[i:i+block_size] for i in ix])
    #在x后移一位，形成对应目标数值(Batch_size, block_size)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    #全部转移到GPU上，形状方便做批量计算
    x, y = x.to(device), y.to(device)
    # 后面每一行的X（1，block_size)会通过因果掩码，变为下三角形状的(block_size,block_size),
    #然后进入Transformer，最后整体变为(1, block_size),与其对应的y进行对比。
    return x, y


#测试在2个数据集上的平均loss

#设置为无需计算梯度，更新模型参数，运算更快
@torch.no_grad()
def estimate_loss():
    out = {}
    # 开启评估模式，关闭dropout等训练手法
    model.eval()
    #测试训练集和验证集
    for split in ['train', 'val']:
        #初始化loss数组形状
        losses = torch.zeros(eval_iters)
        #设置抽取次数（eveval_iters)，最后算mean()
        for k in range(eval_iters):
            #切出要求的（batch_size, block_size)
            X, Y = get_batch(split)
            #模型输出此次损失，与预测词的每一个候选答案的分数
            # softmax(logits) 比对真实的y -> loss
            logits, loss = model(X, Y)
            #记录此次损失在losses数组中
            losses[k] = loss.item()
        #循环后求均值，放入字典
        out[split] = losses.mean()
    #切换为训练模式
    model.train()
    return out

#编写单头注意力
class Head(nn.Module):
    #head_size是每个头分到的要处理的词的特征维度
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)
        #做出固定三角矩阵，作为因果掩码
        #buffer注册不会被求导，但可以迁移，保存入字典
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        
        #获得input(batch, time_step, channels)
        # channels = head_size * num_heads
        
        #batch：批次，一次GPU里算多少句子
        #time-step:时间步，每个句子里多少词输入
        #channels:通道/特征维度，一个词语用多少属性描述
        B,T,C = x.shape
        #线性层映射出对应的k,q,v
        k = self.key(x) # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        v = self.value(x) # (B,T,hs)
        
        # 缩放点积注意力，wei就是weight
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        # 加入因果掩码，GPT是将所有文本随机按固定长度切除，不需要padding
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        # softmax将每个位置的候选词打注意力分数
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        # dropout暂退，增强模型泛化
        wei = self.dropout(wei)
        #利用经历自注意力后的分数更新v的数值，让v之间建立相关性连接
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        #输出out(batch, time-step, head-size)
        return out

#实现多头注意力
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        #创建num_heads个读取heads_size的注意力头
        #用ModuleList解决原生数组不能寻找到权重，更新参数与无法迁移GPU的问题
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        #创建一个线性投影层，对所有头的结果做运算，总结
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        #防止过拟合，增强泛化
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        #将输入的数值经过所有注意力头处理,并合并拼接
        #获得 num_heads * (Batch, num-step, heads_size)
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        #将合并的数值进行整体运算，提炼出整体之间的特征关系
        out = self.dropout(self.proj(out))
        #out(Batch, num-step, n_embd)
        return out

#编写前馈神经网络，一个简单的MLP
class FeedFoward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

#编写单层编码器，GPT是decode-only架构
#多层注意力 + 前馈神经网络 + 归一化与残差连接
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        #sa即为 Self-Attention
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        #创建2个层归一化，保持了参数独立
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        #pre-LN处理，先做归一化，再进入模型处理，再残差连接
        #pre-LN使得主运算是+，无归一化打扰，杜绝了梯度消失，爆炸
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x



class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        #词嵌入
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        #位置嵌入  
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        #创建n_layer个block(decode), *为解包操作符，将容器中的block全部拿出，依次传入Sequential 
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        #采用了 Pre-LN，主干道上的数据一直累加，
        #所以在送出最终结果前，需要做最后一次标准归一化收尾
        
        #最后一个归一化处理
        self.ln_f = nn.LayerNorm(n_embd) 
        #最后一个打分头
        self.lm_head = nn.Linear(n_embd, vocab_size)
          
        #遍历所有网络层，进行初始化
        self.apply(self._init_weights)



    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        #词嵌入编码 从idx(Batch, num-step) -> tok_emb(Batch, num-step, n_embd)
        tok_emb = self.token_embedding_table(idx) 
        #位置编码 (T) -> (T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        #将词嵌入和位置编码相加，同时获得词维度特征与位置特征
        # x(Batch, num-step, n_embd)
        x = tok_emb + pos_emb
        # 经过n_layer层decode处理
        x = self.blocks(x)
        #最后一轮归一化
        x = self.ln_f(x)
        #最后一轮的打分头
        # logits = (Batch, num-step, vocab_size)
        logits = self.lm_head(x)

        #拿出打分后的数据算交叉熵
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            #将B，T压缩在一起，相当于不区分句子与句中词语，全部转为一个个字符
            #转化也是因为交叉熵只支持二维形式
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss


    def generate(self, idx, max_new_tokens, temperature=0.7, top_k=10):
        #限定生成字数的数量
        for _ in range(max_new_tokens):
            #限定一次的最大上下文视野（即为实现自注意力的范围）
            #从倒数第 block_size 个字开始，一直截取到最后一个字
            idx_cond = idx[:, -block_size:]
            # 将截取的数据放入forward中训练，self()即调用了forward
            logits, loss = self(idx_cond)
            #对最后一个位置进行预测，从（Batch, num-step, vocab_size)
            #转变为(Batch, vocab_size)

            #改进：加入temperature温度系数，让高频词集中
            logits = logits[:, -1, :] / temperature

            #改进：Top-k 过滤，保留前K个最高概率的候选字符，其他设置为-inf
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            
            #最后一轮只打分，没做softmax
            probs = F.softmax(logits, dim=-1)
            #掷骰子抽签（Multinomial 采样）
            #multinomial 是按概率轮盘抽签
            #概率 70% 的词被抽中的机会大，概率 20% 的词也有机会露脸
            #写出来的句子才有灵性和创造力
            #idx_next(Batch, 1)
            #为（Batch, 1)即可以支持服务器为多个人进行预测
            # num_sample即为每次选择都只选 num_sample个字符
            idx_next = torch.multinomial(probs, num_samples=1)
            #拼接已经生成的文本与刚预测出的文字
            idx = torch.cat((idx, idx_next), dim=1) 
        return idx


#参数初始化函数
    def _init_weights(self, module):
        #判定是不是全连接层：Y = X * W + b,对全连接层参数做初始化
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
            #对词嵌入层参数做初始化
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)



#========================================================================






# 模型准备
model = GPTLanguageModel()
m = model.to(device)

#展示模型参数量（单位：百万）
#m.parameters()获取模型里所有参数
#p.numel()展示每个参数里多少数字
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# 配置优化器，AdamW是标准优化算法
# 比普通的梯度下降更聪明，能自适应调节不同参数的更新速度，且自带权重衰减
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

#训练环节
# iter即为步数，max_iters即为一共训练多少步
#GPT架构因为需要海量数据，大部分只训练一个epoch，step模式更灵活，也符合学习率调度器按步调整的特性
for iter in range(max_iters):
    #每隔eval_interval轮或者最后一轮就做一次测试，算出2种数据集平均损失
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        #打印2种数据集的各自平均损失
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    
    #选取文本（Batch, block_size)
    xb, yb = get_batch('train')
    #算loss和zero_grad可以调换顺序
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

#初始化一个开头输入，进行预测(Batch, num-step) = (1, 1)即为生成一个文章，这个文章开头就一个字符
context = torch.zeros((1, 1), dtype=torch.long, device=device)
#[0]即为拿到生成的(Batch, max_new_tokens) 中的第一个Batch
#tolist将张量转列表，从Tensor转为python普通原生整数列表
#decode将预测的数值转为字符
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

10.788929 M parameters
step 0: train loss 4.2221, val loss 4.2306
step 500: train loss 1.7522, val loss 1.9146
step 1000: train loss 1.3913, val loss 1.6010
step 1500: train loss 1.2649, val loss 1.5221
step 2000: train loss 1.1891, val loss 1.5078
step 2500: train loss 1.1222, val loss 1.4894
step 2999: train loss 1.0702, val loss 1.4839

And fit our coate to the moise of his crown.

KING RICHARD II:
And happin him! the days not to Rapose,
To defend the foul of summer's heart.

QUEEN ELIZABETH:
Who dost thou dost stir thine express the world?

KING RICHARD III:
But in all haste of long of all firsty,
And be not till the faves of itself to hear.

DUKE OF AUMERLE:
No, man, some shall do more son of many men,
Which is lord in the shame of the strengerous words.

KING RICHARD III:
Stand hath not so deliver hath been the hanos are.
Th
